In [2]:
%useLatestDescriptors
%use dataframe
%use kandy

In [13]:
USE{
    dependencies("org.json:json:20250107")
}

In [14]:
import org.json.XML

In [15]:
enum class SeaSector(val simpleName: String) {
    EAST("001"),
    WEST("002"),
    SOUTH("003")
}

In [16]:
val serviceKeyFilePath = "/Volumes/WorkSpace/Notebook/data/OceanMensurationService.json"
val SDATE = "20250228"
val EDATE = "20250228"
val numOfRows = "100"
val maxPage = 500

In [17]:
fun load(path:String, maxPage:Int): AnyFrame {
    val rows = mutableListOf<AnyFrame>()
    var requestPage = 1
    do{
        val pagePath = "$path&pageNo=$requestPage"
        val jsonData = XML.toJSONObject(DataFrame.read(pagePath).toString())
        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
        try {
            val instanceDf = df.get("response").get("body").get("items").get("item").toDataFrame()
            requestPage += 1
            rows.add(instanceDf)
        } catch(e: Exception) {
            print(e.localizedMessage)
            break
        }
    } while (rows.size < maxPage )
    return rows.concat()
}

In [18]:
val serviceInfo = DataRow.readJson(path=serviceKeyFilePath)

In [19]:
val url_East = "${serviceInfo.endpoint_apis}/${serviceInfo.list_apis}?ServiceKey=${serviceInfo.key_apis}&GRU_NAM=${SeaSector.EAST.simpleName}&SDATE=$SDATE&EDATE=$EDATE&numOfRows=$numOfRows"

In [9]:
val url_West = "${serviceInfo.endpoint_apis}/${serviceInfo.list_apis}?ServiceKey=${serviceInfo.key_apis}&GRU_NAM=${SeaSector.WEST.simpleName}&SDATE=$SDATE&EDATE=$EDATE&numOfRows=$numOfRows"

In [10]:
val url_South = "${serviceInfo.endpoint_apis}/${serviceInfo.list_apis}?ServiceKey=${serviceInfo.key_apis}&GRU_NAM=${SeaSector.SOUTH.simpleName}&SDATE=$SDATE&EDATE=$EDATE&numOfRows=$numOfRows"

In [20]:
val rawEastDf = load(url_East,maxPage )

Can not get nested column 'item' from ValueColumn 'items'

In [22]:
val rawEastConcat = rawEastDf.item.concat()

In [23]:
rawEastConcat.head()

staNamKor,cdt_1,obsDtm,staCde,obsTim,wtrTmp_1,gruNam,wtrTmp_3,wtrTmp_2
덕천,34.280000,2025-02-28 23:30:00,bdch3,233000,10.200000,동해,null,null
구룡포 하정,null,2025-02-28 23:30:00,fghe8,233000,10.900000,동해,null,null
고성 가진,null,2025-02-28 23:30:00,fggo3,233000,5.400000,동해,4.500000,5.300000
온양,34.333000,2025-02-28 23:30:00,byyh3,233000,9.800000,동해,null,null
영덕,null,2025-02-28 23:30:00,byd8a,233000,10.400000,동해,10,10.100000


In [30]:
rawEastConcat.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
staNamKor,String,522,12,0,기장,53,null,null,강릉,기장(한수원),진하
cdt_1,Double?,522,150,287,34.328000,5,34.466013,0.312847,34.147000,34.333000,35.184000
obsDtm,String,522,48,0,2025-02-28 23:30:00,12,null,null,2025-02-28 00:00:00,2025-02-28 12:00:00,2025-02-28 23:30:00
staCde,String,522,12,0,bgj8a,53,null,null,bdch3,bngh3,fghe8
obsTim,Comparable<*>,522,48,0,233000,12,null,null,null,null,null
wtrTmp_1,Number,522,54,0,11.700000,52,10.177203,1.845854,5.400000,10.400000,13.400000
gruNam,String,522,1,0,동해,522,null,null,동해,동해,동해
wtrTmp_3,Number?,522,42,278,10,46,8.277869,2.545439,4.500000,8.800000,11.800000
wtrTmp_2,Number?,522,40,278,10,43,8.703279,2.253573,5.100000,9.150000,11.800000


In [33]:
val rawEastParse = rawEastConcat.parse()

In [65]:
rawEastParse
    .select{ staNamKor and wtrTmp_1 }
    .convert( "wtrTmp_1" ).toFloat()
    .sortBy { staNamKor }
    .groupBy{staNamKor}
    .plot{

        layout {
            title = "관측지점별 일평균 해수 정보"
            size = 1400 to 600
            theme = Theme.HIGH_CONTRAST_DARK
            style(Style.Minimal2)
        }

        boxplot("staNamKor", "wtrTmp_1") {
            boxes {
                borderLine.color = Color.BLUE
                fillColor("staNamKor"){
                    scale = categoricalColorHue()
                    legend{
                        name= "관측 지점"
                    }
                }
            }

            x.axis.name = "관측 지점"
            y.axis.name = "수온 °C"
            y.axis.limits = 3.0..15.0
        }

    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="znNlwY"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"관측지점별 일평균 해수 정보"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"flip":false,
"ylim":[3.0,15.0]
},
"data":{
},
"ggsize":{
"width":1400.0,
"height":600.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"staNamKor",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"wtrTmp_1",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"fill",
"scale_mapper_kind":"color_hue",
"name":"관측 지점"
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"name":"관측 지점",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"수온 °C",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"ymin":"min",
"lower":"lower",
"middle":"middle",
"upper":"upper",
"ymax":"max",
"fill":"staNamKor",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"staNamKor":["강릉","고리","고성 가진","구룡포 하정","기장","기장(한수원)","나곡","덕천","삼척","영덕","온양","진하"],
"&merged_groups":["강릉","고리","고성 가진","구룡포 하정","기장","기장(한수원)","나곡","덕천","삼척","영덕","온양","진하"],
"min":[8.199999809265137,11.899999618530273,5.400000095367432,10.399999618530273,11.5,11.600000381469727,9.600000381469727,9.699999809265137,9.100000381469727,10.199999809265137,9.800000190734863,11.800000190734863],
"middle":[8.5,12.5,5.400000095367432,10.800000190734863,11.699999809265137,11.699999809265137,9.899999618530273,10.349999904632568,9.399999618530273,10.399999618530273,9.899999618530273,11.899999618530273],
"max":[8.899999618530273,13.399999618530273,5.599999904632568,11.300000190734863,11.699999809265137,11.699999809265137,11.800000190734863,12.100000381469727,9.699999809265137,10.5,10.0,12.100000381469727],
"lower":[8.399999618530273,12.199999809265137,5.400000095367432,10.5,11.600000381469727,11.600000381469727,9.800000190734863,9.825000047683716,9.325000047683716,10.300000190734863,9.899999618530273,11.899999618530273],
"upper":[8.600000381469727,12.775000095367432,5.5,11.0,11.699999809265137,11.699999809265137,10.600000381469727,10.874999761581421,9.5,10.399999618530273,9.974999904632568,12.0],
"x":["강릉","고리","고성 가진","구룡포 하정","기장","기장(한수원)","나곡","덕천","삼척","영덕","온양","진하"]
},
"color":"#5470c6",
"sampling":"none",
"inherit_aes":false,
"position":{
"name":"dodge",
"width":1.0
},
"geom":"boxplot",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"staNamKor"
},{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"min"
},{
"type":"float",
"column":"lower"
},{
"type":"float",
"column":"middle"
},{
"type":"float",
"column":"upper"
},{
"type":"float",
"column":"max"
},{
"type":"str",
"column":"&merged_groups"
}]
}
},{
"mapping":{
"x":"x",
"y":"y",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["강릉","온양","온양","온양","진하"],
"x":["강릉","온양","온양","온양","진하"],
"y":[9.0,10.199999809265137,10.199999809265137,10.199999809265137,11.699999809265137]
},
"sampling":"none",
"inherit_aes":false,
"position":{
"name":"dodge",
"width":1.0
},
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"staNamKor"
},{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
},{
"type":"str",
"column":"&merged_groups"
}]
}
}],
"theme":{
"name":"minimal2",
"axis_ontop":false,
"axis_ontop_y":false,
"axis_ontop_x":false,
"flavor":"high_contrast_dark"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"staNamKor"
},{
"type":"str",
"column":"&merged_groups"
}]
},
"spec_id":"104"
};
 var containerDiv = document.getElementById("znNlwY");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 1400.0,


In [12]:
val rawWestDf = load(url_West,maxPage )

Can not get nested column 'item' from ValueColumn 'items'

In [16]:
val rawWestConcat = rawWestDf.item.concat()

In [17]:
rawWestConcat.head()

staNamKor,cdt_1,obsDtm,staCde,dox_1,obsTim,wtrTmp_1,gruNam,dox_2,wtrTmp_2
태안 고남,31.200000,2025-02-27 23:30:00,br001,11.700000,233000,4,서해,null,null
서산 지곡,null,2025-02-27 23:30:00,sj086,null,233000,4.200000,서해,null,null
태안 파도리,null,2025-02-27 23:30:00,ftpk5,null,233000,4.700000,서해,14.100000,4.600000
태안 대야도,null,2025-02-27 23:30:00,ftdk5,12.500000,233000,4.100000,서해,null,4.100000
서산 창리,31.600000,2025-02-27 23:30:00,fsch6,13,233000,3.600000,서해,null,3.300000


In [13]:
val rawSouthDf = load(url_South,maxPage )

Can not get nested column 'item' from ValueColumn 'items'

In [18]:
val rawSouthConcat = rawSouthDf.item.concat()

In [19]:
rawSouthConcat.head()

staNamKor,obsDtm,staCde,dox_1,obsTim,wtrTmp_1,gruNam,wtrTmp_2,cdt_1,wtrTmp_3,dox_3,dox_2
장흥 회진,2025-02-27 23:30:00,ejhfc,12.200000,233000,7.300000,남해,7.300000,null,null,null,null
완도 노화도,2025-02-27 23:30:00,wn087,null,233000,6.700000,남해,null,null,null,null,null
완도 금일,2025-02-27 23:30:00,wk094,null,233000,7.600000,남해,null,null,null,null,null
완도 청산,2025-02-27 23:30:00,wc001,10.100000,233000,8.400000,남해,null,null,null,null,null
통영 사량,2025-02-27 23:30:00,ty005,10.800000,233000,7.100000,남해,null,null,null,null,null


In [20]:
val rawDf = listOf(rawEastConcat, rawWestConcat, rawSouthConcat).concat()

In [21]:
rawDf.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
staNamKor,String,4377,143,0,남해 강진,72,null,null,강릉,여수 신월,해남 황산
cdt_1,Number?,4377,156,3849,34.800000,52,33.670780,1.270628,30.700000,34.297000,35.257000
obsDtm,String,4377,48,0,2025-02-27 18:00:00,141,null,null,2025-02-27 00:00:00,2025-02-27 12:00:00,2025-02-27 23:30:00
staCde,String,4377,143,0,eng5c,72,null,null,bdch3,fhyk7,wn087
obsTim,Comparable<*>?,4377,49,2060,000000,56,null,null,null,null,null
wtrTmp_1,Comparable<*>?,4377,109,8,6.200000,147,null,null,null,null,null
gruNam,String,4377,3,0,남해,2887,null,null,남해,남해,서해
wtrTmp_3,Number?,4377,43,3987,9.200000,33,8.027436,2.152572,4.600000,8.050000,11.900000
wtrTmp_2,Number?,4377,70,3379,9.200000,53,7.045391,2.030518,3.200000,6.900000,11.900000
dox_1,Number?,4377,55,3472,10.200000,79,11.368508,1.444341,8.900000,11.300000,14.500000


In [22]:
val rawDfParse = rawDf.parse()

In [197]:
rawDfParse.filter{gruNam.equals("동해")}
    .drop{ staNamKor in dropList }
    .select{ staNamKor and wtrTmp_1 }
    .convert( "wtrTmp_1" ).toFloat()
    .sortBy { staNamKor }
    .plot{

        layout {
            title = "관측지점별 일평균 해수 정보"
            size = 1400 to 600
        //    theme = Theme.HIGH_CONTRAST_DARK
        }

        boxplot("staNamKor", "wtrTmp_1") {
            boxes {
                borderLine.color = Color.BLUE
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="Ybthcq"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"관측지점별 일평균 해수 정보"
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":1400.0,
"height":600.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"staNamKor",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"wtrTmp_1",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"ymin":"min",
"lower":"lower",
"middle":"middle",
"upper":"upper",
"ymax":"max"
},
"stat":"identity",
"data":{
"middle":[7.199999809265137,12.5,5.400000095367432,10.399999618530273,11.699999809265137,11.699999809265137,10.300000190734863,10.399999618530273,9.699999809265137,10.300000190734863,9.899999618530273,11.800000190734863],
"min":[7.0,12.0,5.300000190734863,9.899999618530273,11.5,11.600000381469727,10.0,9.899999618530273,9.399999618530273,10.100000381469727,9.800000190734863,11.600000381469727],
"max":[7.800000190734863,13.5,5.5,10.899999618530273,11.800000190734863,11.899999618530273,10.600000381469727,11.199999809265137,9.899999618530273,10.699999809265137,10.0,11.899999618530273],
"lower":[7.099999904632568,12.199999809265137,5.400000095367432,10.0,11.600000381469727,11.699999809265137,10.199999809265137,10.199999809265137,9.600000381469727,10.199999809265137,9.800000190734863,11.699999809265137],
"upper":[7.400000095367432,12.800000190734863,5.475000023841858,10.699999809265137,11.800000190734863,11.800000190734863,10.399999618530273,10.625000238418579,9.800000190734863,10.474999904632568,9.899999618530273,11.800000190734863],
"x":["강릉","고리","고성 가진","구룡포 하정","기장","기장(한수원)","나곡","덕천","삼척","영덕","온양","진하"]
},
"color":"#5470c6",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"boxplot",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"min"
},{
"type":"float",
"column":"lower"
},{
"type":"float",
"column":"middle"
},{
"type":"float",
"column":"upper"
},{
"type":"float",
"column":"max"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["강릉","강릉","고리","나곡","나곡","나곡","나곡","덕천","덕천","덕천","덕천","삼척","삼척","삼척","진하","진하"],
"y":[8.199999809265137,8.199999809265137,13.800000190734863,9.899999618530273,9.899999618530273,10.699999809265137,10.699999809265137,11.5,12.0,11.699999809265137,11.600000381469727,9.300000190734863,9.100000381469727,9.300000190734863,12.0,12.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
}],
"spec_id":"209"
};
 var containerDiv = document.getElementById("Ybthcq");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 1400.0,
 height: 600.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 강릉 
 
 
 
 
 
 
 
 
 고리 
 
 
 
 
 
 
 
 
 고성 가진 
 
 
 
 
 


In [198]:
rawDfParse.filter{gruNam.equals("서해")}
    .drop{ staNamKor in dropList }
    .select{ staNamKor and wtrTmp_1 }
    .convert( "wtrTmp_1" ).toFloat()
    .sortBy { staNamKor }
    .plot{

        layout {
            title = "관측지점별 일평균 해수 정보"
            size = 1400 to 600
  //          theme = Theme.HIGH_CONTRAST_DARK
        }

        boxplot("staNamKor", "wtrTmp_1") {
            boxes {
                borderLine.color = Color.BLUE
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="D8CeuZ"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"관측지점별 일평균 해수 정보"
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":1400.0,
"height":600.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"staNamKor",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"wtrTmp_1",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"ymin":"min",
"lower":"lower",
"middle":"middle",
"upper":"upper",
"ymax":"max"
},
"stat":"identity",
"data":{
"min":[3.5999999046325684,4.800000190734863,4.5,3.5999999046325684,3.299999952316284,4.0,4.199999809265137,2.9000000953674316,3.200000047683716,7.5,5.199999809265137,5.400000095367432,7.400000095367432,4.699999809265137,4.199999809265137,4.400000095367432,4.800000190734863,4.800000190734863,4.599999904632568,5.900000095367432,5.5,5.800000190734863,3.9000000953674316,3.799999952316284,3.700000047683716,4.699999809265137,4.800000190734863,5.300000190734863],
"middle":[3.850000023841858,4.900000095367432,4.900000095367432,4.0,4.300000190734863,4.800000190734863,4.599999904632568,3.549999952316284,3.4000000953674316,7.699999809265137,5.699999809265137,5.700000047683716,7.900000095367432,5.549999952316284,4.900000095367432,5.25,5.300000190734863,5.5,5.199999809265137,6.199999809265137,6.599999904632568,5.900000095367432,4.0,4.0,3.9000000953674316,5.75,6.099999904632568,5.699999809265137],
"max":[4.199999809265137,5.199999809265137,5.099999904632568,5.199999809265137,5.800000190734863,6.0,5.199999809265137,4.199999809265137,3.9000000953674316,7.900000095367432,6.400000095367432,6.099999904632568,8.699999809265137,6.599999904632568,5.400000095367432,5.900000095367432,5.900000095367432,6.0,5.5,6.599999904632568,7.599999904632568,6.099999904632568,4.199999809265137,4.199999809265137,4.199999809265137,7.199999809265137,6.800000190734863,6.0],
"upper":[4.0,5.0,5.0,4.450000047683716,5.400000095367432,5.699999809265137,4.900000095367432,3.774999976158142,3.6249999403953552,7.800000190734863,6.0,5.925000071525574,8.150000095367432,6.074999928474426,5.0,5.5,5.475000023841858,5.8500001430511475,5.300000190734863,6.400000095367432,6.800000190734863,6.0,4.099999904632568,4.099999904632568,4.0,6.524999976158142,6.5,5.800000190734863],
"lower":[3.700000047683716,4.800000190734863,4.599999904632568,3.799999952316284,3.9000000953674316,4.5,4.400000095367432,3.200000047683716,3.299999952316284,7.5,5.549999952316284,5.475000023841858,7.400000095367432,4.800000190734863,4.400000095367432,4.6249998807907104,5.099999904632568,4.950000047683716,4.8750001192092896,6.174999833106995,5.925000071525574,5.900000095367432,3.9000000953674316,3.9000000953674316,3.799999952316284,5.0,5.099999904632568,5.549999952316284],
"x":["군산 신시도","목포","목포 외달","무안 도리포","무안 서북","무안 성내","백령도","서산 지곡","서산 창리","신안 다물도","신안 다수","신안 대리","신안 마리","신안 반월","신안 소신","신안 송공","신안 안좌","신안 장산","신안 하의","진도 가사","진도 옥도","진도 전두","태안 고남","태안 대야도","태안 파도리","해남 궁항","해남 문내","해남 임하"]
},
"color":"#5470c6",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"boxplot",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"min"
},{
"type":"float",
"column":"lower"
},{
"type":"float",
"column":"middle"
},{
"type":"float",
"column":"upper"
},{
"type":"float",
"column":"max"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["목포","목포","진도 가사","진도 가사","진도 전두","진도 전두","진도 전두","진도 전두","진도 전두","태안 파도리","태안 파도리","태안 파도리","태안 파도리","태안 파도리","태안 파도리"],
"y":[5.300000190734863,5.300000190734863,6.800000190734863,6.8

In [199]:
rawDfParse.filter{gruNam.equals("남해")}
    .drop{ staNamKor in dropList }
    .select{ staNamKor and wtrTmp_1 }
    .convert( "wtrTmp_1" ).toFloat()
    .sortBy { staNamKor }
    .plot{

        layout {
            title = "관측지점별 일평균 해수 정보"
            size = 1400 to 600
   //         theme = Theme.HIGH_CONTRAST_DARK
        }

        boxplot("staNamKor", "wtrTmp_1") {
            boxes {
                borderLine.color = Color.BLUE
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="4h3uho"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"관측지점별 일평균 해수 정보"
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":1400.0,
"height":600.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"staNamKor",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"wtrTmp_1",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"ymin":"min",
"lower":"lower",
"middle":"middle",
"upper":"upper",
"ymax":"max"
},
"stat":"identity",
"data":{
"middle":[6.450000047683716,6.699999809265137,8.199999809265137,10.899999618530273,7.099999904632568,5.699999809265137,7.199999809265137,7.800000190734863,6.300000190734863,7.199999809265137,7.099999904632568,7.099999904632568,6.0,4.900000095367432,5.199999809265137,5.199999809265137,5.699999809265137,5.5,5.699999809265137,8.600000381469727,5.0,6.3500001430511475,6.300000190734863,6.25,6.5,6.400000095367432,13.199999809265137,6.3999998569488525,7.699999809265137,5.400000095367432,6.5,10.449999809265137,6.0,6.400000095367432,6.0,5.25,4.599999904632568,7.199999809265137,5.300000190734863,5.699999809265137,4.0,6.8500001430511475,6.5,6.049999952316284,7.150000095367432,6.6499998569488525,6.5,7.5,7.599999904632568,6.0,6.0,7.599999904632568,6.900000095367432,6.800000190734863,7.400000095367432,6.300000190734863,5.800000190734863,7.599999904632568,6.199999809265137,8.199999809265137,7.5,7.599999904632568,7.199999809265137,7.199999809265137,6.599999904632568,8.949999809265137,6.8500001430511475,7.549999952316284,7.599999904632568,6.800000190734863,8.300000190734863,6.5,6.699999809265137,7.300000190734863,4.900000095367432,7.400000095367432,7.1499998569488525,6.700000047683716,6.400000095367432,6.8500001430511475,6.900000095367432,6.5,7.5,7.199999809265137,9.0,8.650000095367432,8.199999809265137,9.199999809265137,7.699999809265137,6.299999952316284,5.900000095367432,5.599999904632568,10.800000190734863,5.599999904632568,5.25,6.0,5.400000095367432,5.300000190734863,5.699999809265137],
"min":[6.099999904632568,6.599999904632568,8.0,10.800000190734863,6.0,4.800000190734863,7.099999904632568,7.5,6.199999809265137,6.5,6.800000190734863,6.800000190734863,5.400000095367432,4.099999904632568,4.099999904632568,5.0,5.300000190734863,4.900000095367432,5.099999904632568,8.600000381469727,3.9000000953674316,5.699999809265137,5.800000190734863,5.800000190734863,5.900000095367432,6.099999904632568,12.699999809265137,4.800000190734863,7.199999809265137,3.5999999046325684,5.900000095367432,9.800000190734863,5.5,5.900000095367432,5.699999809265137,3.700000047683716,4.300000190734863,6.900000095367432,4.900000095367432,5.300000190734863,3.700000047683716,6.400000095367432,6.099999904632568,5.599999904632568,5.800000190734863,6.199999809265137,6.199999809265137,7.099999904632568,7.5,5.5,5.300000190734863,7.5,5.599999904632568,6.699999809265137,6.800000190734863,6.099999904632568,5.300000190734863,7.400000095367432,6.099999904632568,7.900000095367432,7.300000190734863,7.199999809265137,6.800000190734863,6.300000190734863,6.0,8.699999809265137,6.300000190734863,6.900000095367432,7.5,6.300000190734863,7.900000095367432,6.199999809265137,6.199999809265137,7.0,4.900000095367432,6.900000095367432,6.800000190734863,5.800000190734863,5.400000095367432,6.400000095367432,6.5,5.5,7.300000190734863,6.800000190734863,8.699999809265137,8.399999618530273,7.900000095367432,9.199999809265137,6.699999809265137,4.599999904632568,5.300000190734863,5.099999904632568,10.199999809265137,5.099999904632568,4.900000095367432,5.199999809265137,4.599999904632568,

In [56]:
val dropList = rawDfParse.filter {wtrTmp_1 == null }.distinct { staNamKor }.staNamKor.toList()
dropList

[함평 석두, 신안 압해, 완도 동촌, 완도 덕동]

In [66]:
rawDfParse.drop{ staNamKor in dropList }.filter{gruNam.equals("동해")}.convert( "wtrTmp_1" ).toFloat()
    .plot{

        layout {
            title = "관측지점별 해수 정보"
            size = 1400 to 600
        }

        x(obsDtm) { axis.name = "관측일시"}
        y(wtrTmp_1) {axis.name ="표층수온"}
        y.axis.limits = 2.0..15.0
        line{
            color(staNamKor){
             //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측소명"
                }
            }
        }

      //  facetWrap(nRow = 3){ facet(gruNam)   }

}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="8XY8Wb"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"관측지점별 해수 정보"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"flip":false,
"ylim":[2.0,15.0]
},
"data":{
"staNamKor":["덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","기장","구룡포 하정","고성 가진","영덕","삼척","고리","강릉","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장","덕천","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","기장","구룡포 하정","고성 가진","영덕","삼척","나곡","강릉","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","강릉","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","기장","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","강릉","기장(한수원)","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","기장","구룡포 하정","고성 가진","영덕","삼척","나곡","진하","강릉","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","영덕","삼척","나곡","고리","강릉","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","강릉","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","영덕","삼척","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","기장(한수원)","기장","기장","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","강릉","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","기장","구룡포 하정","고성 가진","영덕","삼척","나곡","진하","강릉","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","기장","기장(한수원)","강릉","고리","진하","나곡","삼척","영덕","온양","고성 가진","구룡포 하정"],
"obsDtm":[1.740699E12,1.740699E12,1.740699E12,1.740699E12,1.740699E12,1.740699E12,1.740699E12,1.740699E12,1.740699E12,1.740699E12,1.740699E12,1.740699E12,1.7406972E12,1.7406972E12,1.7406972E12,1.7406972E12,1.7406972E12,1.7406972E12,1.7406972E12,1.7406954E12,1.7

In [67]:
rawDfParse.drop{ staNamKor in dropList }.filter{gruNam.equals("서해")}.convert( "wtrTmp_1" ).toFloat()
    .plot{

        layout {
            title = "관측지점별 해수 정보"
            size = 1400 to 600
        }

        x(obsDtm) { axis.name = "관측일시"}
        y(wtrTmp_1) {axis.name ="표층수온"}
        y.axis.limits = 2.0..15.0
        line{
            color(staNamKor){
                //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측소명"
                }
            }
        }

        //  facetWrap(nRow = 3){ facet(gruNam)   }

    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="61jLQu"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"관측지점별 해수 정보"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"flip":false,
"ylim":[2.0,15.0]
},
"data":{
"staNamKor":["태안 고남","서산 지곡","태안 파도리","태안 대야도","서산 창리","해남 임하","진도 전두","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","신안 하의","신안 소신","신안 송공","신안 마리","신안 장산","신안 대리","신안 다물도","서산 창리","신안 반월","신안 안좌","무안 서북","무안 성내","목포 외달","무안 도리포","진도 옥도","진도 가사","진도 전두","해남 문내","해남 궁항","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","서산 창리","해남 임하","진도 전두","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","신안 하의","신안 소신","신안 송공","신안 마리","신안 장산","신안 대리","신안 다물도","서산 창리","신안 반월","신안 안좌","목포 외달","무안 도리포","진도 옥도","해남 임하","진도 가사","진도 전두","해남 궁항","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","서산 창리","해남 임하","진도 전두","백령도","목포","군산 신시도","태안 고남","목포 외달","무안 도리포","진도 옥도","해남 임하","진도 가사","진도 전두","해남 문내","백령도","목포","군산 신시도","신안 반월","신안 안좌","무안 서북","무안 성내","목포 외달","무안 도리포","진도 옥도","해남 임하","진도 가사","진도 전두","해남 문내","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","서산 창리","해남 임하","진도 전두","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","신안 하의","신안 소신","신안 송공","신안 마리","신안 장산","신안 대리","신안 다물도","신안 다수","서산 창리","신안 반월","신안 안좌","무안 서북","무안 성내","목포 외달","무안 도리포","진도 옥도","해남 임하","진도 가사","진도 전두","해남 문내","해남 궁항","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","서산 창리","해남 임하","진도 전두","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","신안 하의","신안 소신","신안 송공","신안 마리","신안 장산","신안 대리","신안 다물도","신안 다수","서산 창리","신안 반월","신안 안좌","무안 서북","무안 성내","목포 외달","무안 도리포","진도 옥도","해남 임하","진도 가사","진도 전두","해남 문내","해남 궁항","백령도","목포","군산 신시도","태안 고남","군산 신시도","태안 파도리","태안 대야도","서산 창리","해남 임하","진도 전두","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","신안 하의","신안 소신","신안 송공","신안 마리","신안 장산","신안 대리","신안 다물도","신안 다수","서산 창리","신안 반월","신안 안좌","무안 서북","무안 성내","목포 외달","무안 도리포","진도 옥도","해남 임하","진도 가사","진도 전두","해남 문내","해남 궁항","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","서산 창리","해남 임하","진도 전두","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","신안 하의","신안 소신","신안 송공","신안 마리","신안 장산","신안 대리","신안 다물도","신안 다수","서산 창리","신안 반월","신안 안좌","무안 서북","무안 성내","목포 외달","무안 도리포","진도 옥도","해남 임하","진도 가사","진도 전두","해남 문내","해남 궁항","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","서산 창리","해남 임하","진도 전두","백령도","목포","군산 신시도","태안 고남","진도 전두","해남 문내","해남 궁항","백령도","목포","군산 신시도","신안 대리","신안 다물도","신안 다수","서산 창리","신안 반월","신안 안좌","무안 서북","무안 성내","목포 외달","무안 도리포","진도 옥도","해남 임하","진도 가사","진도 전두","해남 문내","해남 궁항","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","서산 창리","해남 임하","진도 전두","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","신안 하의","신안 소신","신안 송공","신안 마리","신안 장산","신안 대리","신안 다물도","신안 다수","서산 창리","신안 반월","신안 안좌","무안 서북","무안 성내","목포 외달","무안 도리포","진도 옥도","해남 임하","진도 가사","진도 전두","해남 문내","해남 궁항","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","서산 창리","해남 임하","진도 전두","백령도","목포","군산 신시도","태안 고남","태안 대야도","신안 하의","신안 소신","신안 송공","신안 마리","신안 장산","신안 대리","신안 다물도","신안 다수","서산 창리","신안 반월","신안 안좌","무안 서북","무안 성내","목포 외달","무안 도리포","진도 옥도","해남 임하","진도 가사","진도 전두","해남 문내","해남 궁항","백령도","목포","군산 신시도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","서산 창리","해남 임하","진도 전두","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","신안 하의","신안 소신","신안 송공","신안 마리","신안 장산","신안 대리","신안 다물도","신안 다수","서산 창리","신안 반월","신안 안좌","무안 서북","무안 성내","목포 외달","무안 도리포","진도 옥도","해남 임하","진도 가사","진도 전두","해남 문내","해남 궁항","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","서산 창리","해남 임하","진도 전두","백령도","목포","군산 신시도","태안 고남","서산 지곡","태안 파도리","태안 대야도","신안 하의","신안 소신","신안 송공","신안 마리","신안 장산","신안 대리","신안 다물도","신안 다수","서산 창리","신안 반월","신안 안좌","무안 서북","무안 성내","목포 외달","

In [68]:
rawDfParse.drop{ staNamKor in dropList }.filter{gruNam.equals("남해")}.convert( "wtrTmp_1" ).toFloat()
    .plot{

        layout {
            title = "관측지점별 해수 정보"
            size = 1400 to 600
        }

        x(obsDtm) { axis.name = "관측일시"}
        y(wtrTmp_1) {axis.name ="표층수온"}
        y.axis.limits = 2.0..15.0
        line{
            color(staNamKor){
                //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측소명"
                }
            }
        }

        //  facetWrap(nRow = 3){ facet(gruNam)   }

    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="61sxEM"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"관측지점별 해수 정보"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"flip":false,
"ylim":[2.0,15.0]
},
"data":{
"staNamKor":["장흥 회진","완도 노화도","완도 금일","완도 청산","통영 사량","통영 영운","통영 비산도","여수 신월","거제 일운","완도 감목","완도 사동","완도 망남","완도 일정","완도 가교","완도 대창","완도 동백","완도 백도","통영 수월","통영 풍화","통영 학림","남해 미조","해남 화산","고흥 소록도","거제 가배","남해 강진","서제주","장흥 회진","여수 봉전","여수 화태","여수 화산","여수 항대","여수 동두","여수 돌산","여수 대율","여수 덕촌","여수 백야","여수 나발","완도 감목","완도 양지","완도 예송","완도 사동","완도 송곡","완도 내리","완도 모동","완도 미라","완도 망남","완도 중도","완도 당인","완도 일정","완도 회룡","완도 가학","완도 고마","완도 군외","완도 가교","완도 대창","완도 동백","완도 방축","완도 백도","완도 당목","통영 수월","통영 풍화","통영 학림","사천 월등2","사천 비토2","남해 미조","장흥 노력","진도 모도","장흥 이진목","진도 회동","진도 금갑","진도 도목","해남 어란","해남 송지","해남 상마","해남 송호","해남 옥동","해남 남성","해남 황산","해남 학가","해남 화산","해남 북일","고흥 염포","고흥 영남","고흥 연소","고흥 시산","고흥 소록도","고흥 발포","고흥 남열","강진 마량","고흥 월정","고흥 지죽","고흥 익금","고흥 금산","거제 가배","고흥 동촌","보성 율포","남해 강진","서제주","고흥 시산","고흥 소록도","고흥 발포","고흥 남열","강진 마량","고흥 월정","고흥 지죽","고흥 익금","고흥 금산","거제 가배","고흥 동촌","보성 율포","남해 강진","서제주","장흥 회진","완도 노화도","완도 금일","완도 청산","통영 사량","통영 영운","통영 비산도","여수 신월","거제 일운","완도 감목","완도 사동","완도 망남","완도 일정","완도 가교","완도 대창","완도 동백","완도 백도","통영 수월","통영 풍화","통영 학림","남해 미조","해남 화산","고흥 소록도","거제 가배","남해 강진","서제주","장흥 회진","완도 사동","완도 송곡","완도 내리","완도 모동","완도 미라","완도 망남","완도 중도","완도 당인","완도 일정","완도 회룡","완도 가학","완도 고마","완도 군외","완도 가교","완도 대창","완도 동백","완도 백도","완도 당목","통영 수월","통영 풍화","통영 학림","사천 월등2","사천 월등1","사천 비토2","사천 비토3","남해 미조","장흥 노력","진도 모도","장흥 이진목","진도 회동","진도 금갑","진도 도목","해남 삼정","해남 송지","해남 상마","해남 송호","해남 옥동","해남 남성","해남 황산","해남 학가","해남 화산","해남 북일","고흥 염포","고흥 영남","고흥 연소","고흥 월하","고흥 소록도","고흥 발포","고흥 남열","강진 마량","고흥 월정","고흥 익금","거제 가배","고흥 동촌","보성 율포","남해 강진","서제주","해남 삼정","해남 송지","해남 상마","해남 송호","해남 옥동","해남 남성","해남 황산","해남 학가","해남 화산","해남 북일","고흥 염포","고흥 영남","고흥 연소","고흥 월하","고흥 소록도","고흥 발포","고흥 남열","강진 마량","고흥 월정","고흥 익금","거제 가배","고흥 동촌","보성 율포","남해 강진","서제주","장흥 회진","완도 노화도","완도 금일","완도 청산","통영 사량","통영 영운","통영 비산도","여수 신월","거제 일운","완도 감목","완도 사동","완도 망남","완도 일정","완도 가교","완도 대창","완도 동백","완도 백도","통영 수월","통영 풍화","통영 학림","남해 미조","해남 화산","고흥 소록도","거제 가배","남해 강진","서제주","장흥 회진","완도 동백","완도 방축","완도 백도","완도 당목","통영 수월","통영 풍화","통영 학림","사천 월등2","사천 비토2","사천 비토1","남해 미조","장흥 노력","진도 모도","장흥 이진목","진도 회동","진도 금갑","진도 도목","해남 어란","해남 삼정","해남 송지","해남 상마","해남 송호","해남 옥동","해남 남성","해남 황산","해남 학가","해남 화산","해남 북일","고흥 염포","고흥 영남","고흥 연소","고흥 월하","고흥 시산","고흥 소록도","고흥 발포","고흥 남열","강진 마량","고흥 월정","고흥 지죽","고흥 익금","고흥 금산","거제 가배","고흥 동촌","보성 율포","남해 강진","서제주","완도 당목","통영 수월","통영 풍화","통영 학림","사천 월등2","사천 비토2","사천 비토1","남해 미조","장흥 노력","진도 모도","장흥 이진목","진도 회동","진도 금갑","진도 도목","해남 어란","해남 삼정","해남 송지","해남 상마","해남 송호","해남 옥동","해남 남성","해남 황산","해남 학가","해남 화산","해남 북일","고흥 염포","고흥 영남","고흥 연소","고흥 월하","고흥 시산","고흥 소록도","고흥 발포","고흥 남열","강진 마량","고흥 월정","고흥 지죽","고흥 익금","고흥 금산","거제 가배","고흥 동촌","보성 율포","남해 강진","서제주","장흥 회진","완도 노화도","완도 금일","완도 청산","통영 사량","통영 영운","통영 비산도","여수 신월","거제 일운","완도 감목","완도 사동","완도 망남","완도 일정","완도 가교","완도 대창","완도 동백","완도 백도","통영 수월","통영 풍화","통영 학림","남해 미조","해남 화산","고흥 소록도","거제 가배","남해 강진","서제주","장흥 회진","진도 도목","해남 어란","해남 삼정","해남 송지","해남 상마","해남 송호","해남 옥동","해남 남성","해남 황산","해남 학가","해남 화산","해남 북일","고흥 염포","고흥 영남","고흥 연소","고흥 월하","고흥 시산","고흥 소록도","고흥 발포","고흥 남열","강진 마량","고흥 월정","고흥 지죽","고흥 익금","고흥 금산","거제 가배","고흥 동촌","보성 율포","남해 강진","서제주","완도 송곡","완도 신흥","완도 내리","완도 모동","완도 미라","완도 망남","완도 중도","완도 당인","완도 일정","완도 회룡","완도 가학","완도 고마","완도 군외","완도 가교","완도 대창","완도 동백","완도 방축","완도 백도","완도 당목","통영 풍화","통영 학림","사천 월등2","사천 월등1","사천 비토2","사천 비토3","남해 미조","장흥 노력","장흥 내저","진도 모도","장흥 이진목","진도 회동","진도 금갑","진도 도목","해남 어란","해남 

In [105]:
val url_East_Month = "${serviceInfo.endpoint_apis}/${serviceInfo.list_apis}?ServiceKey=${serviceInfo.key_apis}&GRU_NAM=${SeaSector.EAST.simpleName}&SDATE=20250221&EDATE=20250227&numOfRows=100"

In [106]:
val rawEastMonthDf = load(url_East_Month, 200 )

Can not get nested column 'item' from ValueColumn 'items'

In [107]:
val rawEastMonthConcat = rawEastMonthDf.item.concat()

In [108]:
rawEastMonthConcat.writeJson("/Volumes/WorkSpace/Notebook/data/21_27.json")

In [3]:
val rawEastMonthConcat = DataFrame.readJson("/Volumes/WorkSpace/Notebook/data/21_27.json")

In [9]:
val rawEastMonthParse = rawEastMonthConcat.parse()

In [5]:
rawEastMonthParse.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
staNamKor,String,3955,13,0,기장,362,null,null,강릉,나곡,진하
cdt_1,Double?,3955,339,2264,34.305000,21,34.487417,0.321047,34.213000,34.343000,35.345000
obsDtm,kotlinx.datetime.LocalDateTime,3955,336,0,2025-02-26T20:30,13,null,null,2025-02-21T00:00,2025-02-24T11:00,2025-02-27T23:30
staCde,String,3955,13,0,bgj8a,362,null,null,bdch3,bngh3,fghe8
obsTim,Int,3955,48,0,203000,90,116259.924147,68934.903643,0,120000,233000
wtrTmp_1,Double,3955,73,0,12.100000,154,9.903110,2.325180,5.200000,10.500000,13.800000
gruNam,String,3955,1,0,동해,3955,null,null,동해,동해,동해
wtrTmp_3,Double?,3955,75,1999,5.300000,100,8.249744,2.550042,4.600000,8.500000,12.500000
wtrTmp_2,Double?,3955,60,1999,5.300000,141,8.396217,2.492433,5.100000,8.900000,12.500000


In [10]:
val rawEastMonthParse2 = rawEastMonthParse.convert( "wtrTmp_1" ).toDouble()
    .add{
        "측정일" from obsDtm.map { it.dayOfMonth }
    }

rawEastMonthParse2.head()

staNamKor,cdt_1,obsDtm,staCde,obsTim,wtrTmp_1,gruNam,wtrTmp_3,wtrTmp_2,측정일
덕천,34.298000,2025-02-27T23:30,bdch3,233000,11.100000,동해,null,null,27
구룡포 하정,null,2025-02-27T23:30,fghe8,233000,10.400000,동해,null,null,27
고성 가진,null,2025-02-27T23:30,fggo3,233000,5.400000,동해,4.900000,5.300000,27
온양,34.359000,2025-02-27T23:30,byyh3,233000,10.000000,동해,null,null,27
영덕,null,2025-02-27T23:30,byd8a,233000,10.400000,동해,10.000000,10.200000,27


In [11]:
rawEastMonthParse2.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
staNamKor,String,3955,13,0,기장,362,null,null,강릉,나곡,진하
cdt_1,Double?,3955,339,2264,34.305000,21,34.487417,0.321047,34.213000,34.343000,35.345000
obsDtm,kotlinx.datetime.LocalDateTime,3955,336,0,2025-02-26T20:30,13,null,null,2025-02-21T00:00,2025-02-24T11:00,2025-02-27T23:30
staCde,String,3955,13,0,bgj8a,362,null,null,bdch3,bngh3,fghe8
obsTim,Int,3955,48,0,203000,90,116259.924147,68934.903643,0,120000,233000
wtrTmp_1,Double,3955,73,0,12.100000,154,9.903110,2.325180,5.200000,10.500000,13.800000
gruNam,String,3955,1,0,동해,3955,null,null,동해,동해,동해
wtrTmp_3,Double?,3955,75,1999,5.300000,100,8.249744,2.550042,4.600000,8.500000,12.500000
wtrTmp_2,Double?,3955,60,1999,5.300000,141,8.396217,2.492433,5.100000,8.900000,12.500000
측정일,Int,3955,7,0,23,576,23.971176,1.986214,21,24,27


In [12]:
rawEastMonthParse2.groupBy { staNamKor and "측정일" }
    .aggregate {
        min{wtrTmp_1} into "min"
        max{wtrTmp_1} into "max"
        mean{wtrTmp_1} into "mean"
    }
    .plot{
        layout {
            title = "관측지점별 일별 해수 정보"
            size = 800 to 400
        }

        ribbon {

            x("측정일") {
                axis.name = "측정일"
            }

            y.axis {
                limits = 2.0..15.0
                name = "수온 최저~최고"
            }

            yMin("min")
            yMax("max")
            fillColor("staNamKor"){
                legend {
                    name = "관측지점"
                }
            }
            alpha = 0.6
            borderLine.width = 0.0

            tooltips(title = value(staNamKor)){
                line("최저 ${value("min")}, 최고 ${value("max")}")
                //line(day, format = "d")
                varLine("측정일", "{d}")
            }
        }


    }



<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="VavayE"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"관측지점별 일별 해수 정보"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"flip":false,
"ylim":[2.0,15.0]
},
"data":{
"staNamKor":["덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","양양","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장"],
"min":[9.9,9.9,5.3,9.8,10.1,9.1,9.9,11.6,12.0,7.0,11.6,11.5,9.9,10.2,5.2,9.8,10.1,9.0,9.9,11.8,12.1,6.7,11.8,11.5,5.3,10.4,9.6,5.3,10.0,5.3,10.1,8.9,10.1,11.5,12.2,6.4,12.0,11.8,10.3,9.4,5.4,10.4,5.4,10.3,9.0,9.5,11.4,12.0,6.6,12.4,11.9,10.4,9.8,5.4,10.7,5.5,10.6,9.1,9.8,11.6,11.9,6.7,11.7,11.4,9.9,10.0,5.5,10.4,5.7,10.8,9.4,9.8,11.2,12.0,6.8,11.9,11.7,10.8,10.0,5.7,10.3,5.5,11.0,9.7,10.4,11.4,11.9,7.0,12.1,11.9],
"max":[12.0,10.9,5.5,10.0,10.7,9.9,10.7,12.0,13.8,8.2,11.9,11.8,12.7,10.9,5.6,10.3,10.2,10.1,11.9,12.2,13.8,7.4,12.2,12.1,5.5,11.6,10.6,5.5,10.4,5.5,10.3,9.6,11.1,12.2,13.6,7.2,12.4,12.3,12.5,10.7,5.7,10.8,5.6,10.6,9.6,11.9,12.0,13.0,7.1,12.7,12.5,11.5,10.5,5.6,10.9,5.7,10.8,9.5,11.1,12.2,12.9,7.4,12.8,12.6,12.9,11.1,5.8,11.2,5.9,11.1,9.8,11.0,12.3,12.9,7.6,12.2,12.0,13.1,10.9,5.9,11.1,5.9,11.3,10.2,12.0,11.7,12.9,7.1,12.4,12.2],
"측정일":[27.0,27.0,27.0,27.0,27.0,27.0,27.0,27.0,27.0,27.0,27.0,27.0,26.0,26.0,26.0,26.0,26.0,26.0,26.0,26.0,26.0,26.0,26.0,26.0,26.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,24.0,24.0,24.0,24.0,24.0,24.0,24.0,24.0,24.0,24.0,24.0,24.0,24.0,23.0,23.0,23.0,23.0,23.0,23.0,23.0,23.0,23.0,23.0,23.0,23.0,23.0,22.0,22.0,22.0,22.0,22.0,22.0,22.0,22.0,22.0,22.0,22.0,22.0,22.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0]
},
"ggsize":{
"width":800.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"y",
"name":"수온 최저~최고",
"limits":[null,null]
},{
"aesthetic":"x",
"name":"측정일",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true,
"name":"관측지점"
}],
"layers":[{
"mapping":{
"x":"측정일",
"ymin":"min",
"ymax":"max",
"fill":"staNamKor"
},
"stat":"identity",
"size":0.0,
"sampling":"none",
"alpha":0.6,
"inherit_aes":false,
"position":"identity",
"geom":"ribbon",
"tooltips":{
"formats":[{
"field":"측정일",
"format":"{d}"
}],
"title":"@staNamKor",
"lines":["최저 @min, 최고 @max","@|@{측정일}"],
"disable_splitting":true
},
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"int",
"column":"측정일"
},{
"type":"float",
"column":"min"
},{
"type":"float",
"column":"max"
},{
"type":"str",
"column":"staNamKor"
}]
},
"spec_id":"5"
};
 var containerDiv = document.getElementById("VavayE");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 800.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 <path d="M28.803489417071205 46.915384615384596 L28.803489417071205 46.915384615384596 L124.81512080730886 51.85384615384612 L220.82675219754606 86.4230769230769 L316.83838358778326 61.73076923076917 L412.85001497802045 83.95384615384614 L508.86164636825765 56.792307